# Query DSL — operator tour

This notebook walks through every operator in the query DSL
against the live corpus. Each section runs a small batch of
queries and tabulates the matching clip counts so you can see
what each operator does.

> New to the DevKit? Start with `01_quickstart.ipynb` for an
> introduction to the dataset and the DSL's place in it.

### Grammar in one screen

| form | meaning |
|---|---|
| `<entity>.<attr> <cmp> <value>` | basic attribute predicate |
| `<entity>(<expr>, …)` | same-entity grouping |
| `and`, `or`, `not` | boolean composition |
| `A while B` | A and B with intersecting intervals |
| `A then(K) B` | B starts during A or within K seconds after |
| `A because_of B` | A's `because_of` edge points at a B |
| `within W: E` | restrict E's time window to W's intervals |

The full specification lives in `docs/query_language.md`.


## Setup


In [ ]:
import os
from pathlib import Path

from causal_ai_av.dataset import CausalAVDataset

# --- shared notebook styling -----------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
    "figure.dpi": 110,
    "savefig.dpi": 110,
})

pd.options.display.max_colwidth = 110
pd.options.display.width = 140
pd.options.display.float_format = "{:.2f}".format


In [ ]:
ds = CausalAVDataset(Path(os.environ["CAUSAL_AV_DATASET_ROOT"]))

def run(queries: list[str]) -> pd.DataFrame:
    """Run a batch of DSL queries and tabulate clip counts."""
    return pd.DataFrame(
        [(q, ds.count(q)) for q in queries],
        columns=["query", "matching clips"],
    )


## Attribute predicates

The atomic predicate is `entity.attribute <cmp> value`. The
comparison is `=` for equality and `<`, `<=`, `>`, `>=` for
ordered attributes.


In [ ]:
run([
    "agent.type = ped",
    "ego.action = decel",
    "light.color = red",
    "env.lanes >= 2",
])


## Hierarchical aliases

Several attribute values are *parent aliases* that expand to a
union of children: `vehicle` (car, truck, bus, …),
`vru` (pedestrian, cyclist), `intersection` (4-way, T-junction,
roundabout, …), `turn` (left, right), `change_lane`
(change-left, change-right).


In [ ]:
run([
    "agent.type = vehicle",
    "agent.type = vru",
    "agent.type = cyclist",
    "env.type = intersection",
])


## Set membership

`attr in (a, b, c)` is sugar for an OR over the listed values.


In [ ]:
run([
    "ego.action in (stop, yield, decel)",
    "agent.type in (ped, cyclist, animal)",
])


## Same-entity coupling

Inside an `agent(…)` clause, every constraint applies to the
**same** agent. Free-floating predicates may bind to different
agents — note how the count changes.


In [ ]:
run([
    "agent(type = vehicle, pos = front)",
    "agent.type = vehicle and agent.pos = front",
])


## Boolean operators

Standard `and`, `or`, `not` over any sub-expression.


In [ ]:
run([
    "agent.type = ped and env.type = crosswalk",
    "ego.action = stop or ego.action = yield",
    "agent.type = ped and not env.type = crosswalk",
])


## Action flags

Action types in the corpus encode flags as parenthesized
suffixes (`oxd:Walk (jaywalk)`). Flag attributes match either
the schema flag field **or** the suffix token.


In [ ]:
run([
    "agent(type = ped, action(jaywalk = true))",
    "agent(action(erratic = true))",
])


## Temporal — `while`

`A while B` keeps only pairs of matches whose intervals
intersect.


In [ ]:
run([
    "agent.type = ped while ego.action = decel",
    "light.color = red while ego.action = stop",
])


## Temporal — `then(K)`

`A then(K) B` keeps pairs where B starts during A or within
K seconds after A ends. Bare `then` (no parens) defaults to
K = 0 — touch or overlap only.


In [ ]:
run([
    "light.color = yellow then(3) ego.action = stop",
    "light.color = green then ego.action = drive",
])


## Relational — `because_of`

Follows the schema's `because_of` causal edge. Both halves are
emitted as match tuples so you can inspect cause and effect.


In [ ]:
run([
    "ego.action = decel because_of agent.type = ped",
    "ego.action = drive because_of agent.type = ped",
])


## Scoping — `within W: E`

Restricts `E`'s evaluation window to `W`'s intervals. Inside,
`not E` means *E does not happen during W* — useful for
negative scenarios like "during red, ego never stops."


In [ ]:
run([
    "within light.color = red: not ego.action = stop",
    "within env.type = crosswalk: agent.type = ped",
])


## Annotator-tagged flags

Some attributes are not derived from kinematics but tagged by
the annotator (e.g. *"the ego could have safely cleared this
yellow"*). They compose with the same DSL.


In [ ]:
run([
    "light(color = yellow, ego_in_on_yellow = true)",
    "light(color = yellow, could_have_cleared = true)",
])
